# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [78]:
# TODO: Fill this in before submitting.
GROUP_ID = 14
NOTEBOOK_URL = "https://github.com/adecku/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Adrian Kawczyński / 318786",
    "Sylwester Kałucki / 312628",
    "Jakub Jarząbek    / 348490"
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [ ]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

In [80]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.11.7
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 20
RAM GiB: 15.71


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [81]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Marketplace listings',
 'feature': 'prices',
 'stress': 'quantiles and category statistics'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [82]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "debug"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 14 {'name': 'Marketplace listings', 'feature': 'prices', 'stress': 'quantiles and category statistics'}
Rows: 200000
Run seed recorded in manifest: 255603670156044160291907649462769696917
Output directory: ..\data\phase2_26L\group_14


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [83]:
# TODO: Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


MARKETPLACE_CATEGORIES = [
    (1, "Electronics", "hard_goods", 1.35, 0.080),
    (2, "Home & Garden", "hard_goods", 0.95, 0.070),
    (3, "Fashion", "soft_goods", 0.55, 0.120),
    (4, "Books", "media", 0.25, 0.060),
    (5, "Sports", "lifestyle", 0.75, 0.090),
    (6, "Beauty", "soft_goods", 0.45, 0.110),
]


def customize_for_variant(df, card, rng):
    n = df.height

    category_ids = rng.choice(
        np.arange(1, len(MARKETPLACE_CATEGORIES) + 1),
        size=n,
        p=[0.24, 0.18, 0.22, 0.12, 0.14, 0.10],
    )

    category_names = np.array([row[1] for row in MARKETPLACE_CATEGORIES])
    category_groups = np.array([row[2] for row in MARKETPLACE_CATEGORIES])
    price_factors = np.array([row[3] for row in MARKETPLACE_CATEGORIES])

    base_price = rng.lognormal(mean=4.2, sigma=0.9, size=n) * price_factors[category_ids - 1]
    discount_pct = rng.choice([0, 5, 10, 15, 20, 30], size=n, p=[0.50, 0.12, 0.16, 0.10, 0.08, 0.04])

    return (
        df.rename({"entity_id": "seller_id", "metric_1": "listing_quality_score", "metric_2": "views_7d"})
        .with_columns(
            pl.Series("category_id", category_ids),
            pl.Series("marketplace_category", category_names[category_ids - 1]),
            pl.Series("category_group", category_groups[category_ids - 1]),
            pl.Series("item_condition", rng.choice(["new", "like_new", "used", "refurbished"], size=n)),
            pl.Series("listing_status", rng.choice(["active", "sold", "expired"], size=n, p=[0.65, 0.25, 0.10])),
            pl.Series("price", base_price.round(2)),
            pl.Series("discount_pct", discount_pct),
            pl.Series("final_price", (base_price * (1 - discount_pct / 100)).round(2)),
        )
    )


def generate_dimension_table(card, rng):
    return pl.DataFrame(
        {
            "category_id": [row[0] for row in MARKETPLACE_CATEGORIES],
            "marketplace_category": [row[1] for row in MARKETPLACE_CATEGORIES],
            "category_group": [row[2] for row in MARKETPLACE_CATEGORIES],
            "expected_price_factor": [row[3] for row in MARKETPLACE_CATEGORIES],
            "commission_rate": [row[4] for row in MARKETPLACE_CATEGORIES],
        }
    )

In [84]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
# events.sort(["event_date", "category"]).write_parquet(
#     OPTIMIZED_EVENTS_PATH,
#     compression="zstd",
#     row_group_size=100_000,
# )

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "created_at_utc": "2026-06-10T19:42:04.136909+00:00",
  "group_id": 14,
  "variant": {
    "name": "Marketplace listings",
    "feature": "prices",
    "stress": "quantiles and category statistics"
  },
  "scale": "debug",
  "rows": 200000,
  "run_seed": 255603670156044160291907649462769696917,
  "paths": {
    "events": "..\\data\\phase2_26L\\group_14\\events.parquet",
    "events_partitioned": "..\\data\\phase2_26L\\group_14\\events_partitioned",
    "events_optimized": "..\\data\\phase2_26L\\group_14\\events_optimized.parquet",
    "dimension": "..\\data\\phase2_26L\\group_14\\dimension.parquet"
  },
  "environment": {
    "python": "3.11.7",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 20,
    "ram_gib": 15.71
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [85]:
print("Rows:", events.height)
print("Columns:", events.columns)

display(events.head(5))

display(events.null_count())

display(
    events.group_by("marketplace_category")
    .agg(
        pl.len().alias("rows"),
        pl.col("final_price").median().round(2).alias("median_price"),
        pl.col("final_price").max().round(2).alias("max_price"),
    )
    .sort("rows", descending=True)
)

display(
    events.group_by("listing_status")
    .agg(pl.len().alias("rows"))
    .sort("rows", descending=True)
)

display(dimension)

Rows: 200000
Columns: ['event_id', 'seller_id', 'event_ts', 'category', 'country', 'device', 'listing_quality_score', 'views_7d', 'tags', 'event_date', 'category_id', 'marketplace_category', 'category_group', 'item_condition', 'listing_status', 'price', 'discount_pct', 'final_price']


event_id,seller_id,event_ts,category,country,device,listing_quality_score,views_7d,tags,event_date,category_id,marketplace_category,category_group,item_condition,listing_status,price,discount_pct,final_price
i64,i64,datetime[μs],str,str,str,f64,i64,list[str],date,i64,str,str,str,str,f64,i64,f64
1,193417,2026-03-20 18:21:55,"""C""","""DE""","""mobile""",26.139,7128,"[""cloud"", ""polars""]",2026-03-20,5,"""Sports""","""lifestyle""","""refurbished""","""active""",44.03,0,44.03
2,1582,2026-01-19 02:28:30,"""A""","""IN""","""mobile""",41.264,8075,"[""security""]",2026-01-19,5,"""Sports""","""lifestyle""","""used""","""active""",28.34,0,28.34
3,148395,2026-02-15 10:22:03,"""B""","""BR""","""mobile""",245.484,5593,"[""ai""]",2026-02-15,2,"""Home & Garden""","""hard_goods""","""like_new""","""active""",173.6,0,173.6
4,110337,2026-01-01 08:48:42,"""F""","""IN""","""mobile""",53.475,2791,"[""etl""]",2026-01-01,1,"""Electronics""","""hard_goods""","""used""","""active""",29.11,0,29.11
5,1413,2026-03-03 02:56:30,"""E""","""UK""","""mobile""",3.408,6981,"[""ai"", ""mlops""]",2026-03-03,1,"""Electronics""","""hard_goods""","""used""","""sold""",41.48,30,29.03


event_id,seller_id,event_ts,category,country,device,listing_quality_score,views_7d,tags,event_date,category_id,marketplace_category,category_group,item_condition,listing_status,price,discount_pct,final_price
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


marketplace_category,rows,median_price,max_price
str,u32,f64,f64
"""Electronics""",47764,83.87,3458.64
"""Fashion""",44352,33.83,1284.85
"""Home & Garden""",35942,59.87,2334.02
"""Sports""",27879,46.74,1386.15
"""Books""",23992,15.47,438.22
"""Beauty""",20071,27.61,1417.6


listing_status,rows
str,u32
"""active""",129937
"""sold""",50153
"""expired""",19910


category_id,marketplace_category,category_group,expected_price_factor,commission_rate
i64,str,str,f64,f64
1,"""Electronics""","""hard_goods""",1.35,0.08
2,"""Home & Garden""","""hard_goods""",0.95,0.07
3,"""Fashion""","""soft_goods""",0.55,0.12
4,"""Books""","""media""",0.25,0.06
5,"""Sports""","""lifestyle""",0.75,0.09
6,"""Beauty""","""soft_goods""",0.45,0.11


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [86]:
# TODO: Implement or adapt benchmark helpers before collecting final results.
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

# TODO: Implement your timing and memory measurement helper.
def path_size_mb(path):
    path = Path(path)

    if path.is_file():
        return path.stat().st_size / 2**20

    if path.is_dir():
        return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / 2**20

    return None


def result_signature(result):
    if isinstance(result, pl.DataFrame):
        result = result.to_pandas()
    elif not isinstance(result, pd.DataFrame):
        result = pd.DataFrame(result)

    comparable = result.copy()
    comparable = comparable.reindex(sorted(comparable.columns), axis=1)

    for col in comparable.columns:
        if pd.api.types.is_numeric_dtype(comparable[col]):
            comparable[col] = comparable[col].astype(float).round(4)
        else:
            comparable[col] = comparable[col].astype(str)

    if len(comparable) > 0:
        comparable = comparable.sort_values(list(comparable.columns)).reset_index(drop=True)

    checksum = int(pd.util.hash_pandas_object(comparable, index=True).sum())
    return f"rows={len(comparable)};cols={len(comparable.columns)};checksum={checksum}"


def benchmark_query(
    library_engine,
    mode,
    query_name,
    query_func,
    input_path=EVENTS_PATH,
    data_format="parquet",
    layout="default",
    repeats=3,
    notes="",
):
    times = []
    memory_values = []
    last_result = None
    process = psutil.Process(os.getpid())

    for _ in range(repeats):
        gc.collect()

        memory_before_mb = process.memory_info().rss / 2**20
        start = time.perf_counter()

        last_result = query_func()

        elapsed = time.perf_counter() - start
        memory_after_mb = process.memory_info().rss / 2**20

        times.append(elapsed)
        memory_values.append(max(memory_before_mb, memory_after_mb))

    row = {
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": N_ROWS,
        "median_time_s": round(float(np.median(times)), 4),
        "peak_memory_mb": round(float(max(memory_values)), 1),
        "input_size_mb": round(float(path_size_mb(input_path)), 2),
        "result_check": result_signature(last_result),
        "notes": notes + " Memory is approximate RSS before/after query.",
    }

    benchmark_results.append(row)
    return row


def benchmark_table():
    return pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)


## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [87]:
QUERY_SPECS = [
    {
        "query_name": "q1_price_stats_active_listings",
        "query_class": "selective filter + category aggregation + quantiles",
        "description": (
            "Filter active marketplace listings from selected countries and compute "
            "price statistics per marketplace category."
        ),
        "hypothesis": (
            "Polars lazy and DuckDB should perform well because they can push filters "
            "and column selection into the Parquet scan. Pandas may use more memory "
            "because it reads data eagerly."
        ),
    },
    {
        "query_name": "q2_top_sellers_by_views",
        "query_class": "high-cardinality group-by + top-k",
        "description": (
            "Group listings by seller_id and find sellers with the largest number "
            "of listings and highest total views."
        ),
        "hypothesis": (
            "This query stresses hash aggregation because seller_id has high cardinality. "
            "Polars and DuckDB should be faster than Pandas. Spark local may have visible "
            "startup and scheduling overhead."
        ),
    },
    {
        "query_name": "q3_commission_by_category_group",
        "query_class": "join with dimension table + aggregation",
        "description": (
            "Join listings with the category dimension table and estimate commission "
            "revenue for sold listings by category group."
        ),
        "hypothesis": (
            "DuckDB should be strong because this query is naturally SQL-shaped and "
            "the dimension table is small. Column pruning should reduce the amount of "
            "data read from Parquet."
        ),
    },
]

pd.DataFrame(QUERY_SPECS)

,query_name,query_class,description,hypothesis
0,q1_price_stats_active_listings,selective filter + category aggregation + quan...,Filter active marketplace listings from select...,Polars lazy and DuckDB should perform well bec...
1,q2_top_sellers_by_views,high-cardinality group-by + top-k,Group listings by seller_id and find sellers w...,This query stresses hash aggregation because s...
2,q3_commission_by_category_group,join with dimension table + aggregation,Join listings with the category dimension tabl...,DuckDB should be strong because this query is ...


### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [88]:
import os
import shutil

print("java path:", shutil.which("java"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

java path: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
JAVA_HOME: None


In [89]:
!java -version

java version "21.0.3" 2024-04-16 LTS
Java(TM) SE Runtime Environment (build 21.0.3+7-LTS-152)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.3+7-LTS-152, mixed mode, sharing)


In [90]:
# Configure Spark local mode for the PySpark benchmark.
spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [91]:
def pandas_read_default(columns):
    return pd.read_parquet(EVENTS_PATH, columns=columns)


def pandas_read_pyarrow(columns):
    return pd.read_parquet(EVENTS_PATH, columns=columns, engine="pyarrow", dtype_backend="pyarrow")


def pandas_q1_price_stats(read_func):
    df = read_func([
        "event_date",
        "country",
        "listing_status",
        "marketplace_category",
        "final_price",
    ])

    result = (
        df[
            (df["listing_status"] == "active")
            & (df["country"].isin(["PL", "DE", "FR"]))
            & (df["event_date"] >= pd.to_datetime("2026-03-01").date())
        ]
        .groupby("marketplace_category", as_index=False)
        .agg(
            listings=("final_price", "size"),
            avg_price=("final_price", "mean"),
            median_price=("final_price", "median"),
            p90_price=("final_price", lambda x: x.quantile(0.90)),
        )
        .sort_values("listings", ascending=False)
    )

    return result


def pandas_q2_top_sellers(read_func):
    df = read_func([
        "seller_id",
        "views_7d",
        "final_price",
    ])

    result = (
        df.groupby("seller_id", as_index=False)
        .agg(
            listings=("seller_id", "size"),
            total_views=("views_7d", "sum"),
            avg_price=("final_price", "mean"),
        )
        .sort_values(["total_views", "listings"], ascending=False)
        .head(20)
    )

    return result


def pandas_q3_commission(read_func):
    events_df = read_func([
        "category_id",
        "listing_status",
        "final_price",
    ])

    dimension_df = pd.read_parquet(DIMENSION_PATH)

    joined = events_df.merge(dimension_df, on="category_id", how="left")
    sold = joined[joined["listing_status"] == "sold"].copy()
    sold["commission_value"] = sold["final_price"] * sold["commission_rate"]

    result = (
        sold.groupby("category_group", as_index=False)
        .agg(
            sold_listings=("final_price", "size"),
            gross_value=("final_price", "sum"),
            commission_value=("commission_value", "sum"),
        )
        .sort_values("commission_value", ascending=False)
    )

    return result

In [92]:
# Pandas implementations of the three benchmark queries
benchmark_query(
    "pandas",
    "default",
    "q1_price_stats_active_listings",
    lambda: pandas_q1_price_stats(pandas_read_default),
    notes="Pandas default backend, selected columns",
)

benchmark_query(
    "pandas",
    "pyarrow",
    "q1_price_stats_active_listings",
    lambda: pandas_q1_price_stats(pandas_read_pyarrow),
    notes="Pandas PyArrow dtype backend, selected columns",
)

benchmark_query(
    "pandas",
    "default",
    "q2_top_sellers_by_views",
    lambda: pandas_q2_top_sellers(pandas_read_default),
    notes="Pandas default backend, selected columns",
)

benchmark_query(
    "pandas",
    "pyarrow",
    "q2_top_sellers_by_views",
    lambda: pandas_q2_top_sellers(pandas_read_pyarrow),
    notes="Pandas PyArrow dtype backend, selected columns",
)

benchmark_query(
    "pandas",
    "default",
    "q3_commission_by_category_group",
    lambda: pandas_q3_commission(pandas_read_default),
    notes="Pandas default backend, selected columns",
)

benchmark_query(
    "pandas",
    "pyarrow",
    "q3_commission_by_category_group",
    lambda: pandas_q3_commission(pandas_read_pyarrow),
    notes="Pandas PyArrow dtype backend, selected columns",
)

benchmark_table()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pandas,default,q1_price_stats_active_listings,parquet,default,200000,0.1421,409.3,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas default backend, selected columns Memor..."
1,pandas,pyarrow,q1_price_stats_active_listings,parquet,default,200000,0.0925,422.5,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas PyArrow dtype backend, selected columns..."
2,pandas,default,q2_top_sellers_by_views,parquet,default,200000,0.1645,431.9,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas default backend, selected columns Memor..."
3,pandas,pyarrow,q2_top_sellers_by_views,parquet,default,200000,0.1898,454.0,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas PyArrow dtype backend, selected columns..."
4,pandas,default,q3_commission_by_category_group,parquet,default,200000,0.1680,458.5,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas default backend, selected columns Memor..."
5,pandas,pyarrow,q3_commission_by_category_group,parquet,default,200000,0.1377,461.9,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas PyArrow dtype backend, selected columns..."


In [93]:
# Polars implementations of the three benchmark queries.

def polars_q1_price_stats_eager():
    df = pl.read_parquet(
        EVENTS_PATH,
        columns=["event_date", "country", "listing_status", "marketplace_category", "final_price"],
    )

    return (
        df.filter(
            (pl.col("listing_status") == "active")
            & (pl.col("country").is_in(["PL", "DE", "FR"]))
            & (pl.col("event_date") >= pl.date(2026, 3, 1))
        )
        .group_by("marketplace_category")
        .agg(
            pl.len().alias("listings"),
            pl.col("final_price").mean().alias("avg_price"),
            pl.col("final_price").median().alias("median_price"),
            pl.col("final_price").quantile(0.90, interpolation="linear").alias("p90_price"),
        )
        .sort("listings", descending=True)
    )


def polars_q1_price_stats_lazy(streaming=False):
    query = (
        pl.scan_parquet(EVENTS_PATH)
        .select(["event_date", "country", "listing_status", "marketplace_category", "final_price"])
        .filter(
            (pl.col("listing_status") == "active")
            & (pl.col("country").is_in(["PL", "DE", "FR"]))
            & (pl.col("event_date") >= pl.date(2026, 3, 1))
        )
        .group_by("marketplace_category")
        .agg(
            pl.len().alias("listings"),
            pl.col("final_price").mean().alias("avg_price"),
            pl.col("final_price").median().alias("median_price"),
            pl.col("final_price").quantile(0.90, interpolation="linear").alias("p90_price"),
        )
        .sort("listings", descending=True)
    )

    if streaming:
        return query.collect(engine="streaming")
    return query.collect()


def polars_q2_top_sellers_eager():
    df = pl.read_parquet(EVENTS_PATH, columns=["seller_id", "views_7d", "final_price"])

    return (
        df.group_by("seller_id")
        .agg(
            pl.len().alias("listings"),
            pl.col("views_7d").sum().alias("total_views"),
            pl.col("final_price").mean().alias("avg_price"),
        )
        .sort(["total_views", "listings"], descending=True)
        .head(20)
    )


def polars_q2_top_sellers_lazy(streaming=False):
    query = (
        pl.scan_parquet(EVENTS_PATH)
        .select(["seller_id", "views_7d", "final_price"])
        .group_by("seller_id")
        .agg(
            pl.len().alias("listings"),
            pl.col("views_7d").sum().alias("total_views"),
            pl.col("final_price").mean().alias("avg_price"),
        )
        .sort(["total_views", "listings"], descending=True)
        .head(20)
    )

    if streaming:
        return query.collect(engine="streaming")
    return query.collect()


def polars_q3_commission_eager():
    events_df = pl.read_parquet(EVENTS_PATH, columns=["category_id", "listing_status", "final_price"])
    dimension_df = pl.read_parquet(DIMENSION_PATH)

    return (
        events_df.join(dimension_df, on="category_id", how="left")
        .filter(pl.col("listing_status") == "sold")
        .with_columns((pl.col("final_price") * pl.col("commission_rate")).alias("commission_value"))
        .group_by("category_group")
        .agg(
            pl.len().alias("sold_listings"),
            pl.col("final_price").sum().alias("gross_value"),
            pl.col("commission_value").sum().alias("commission_value"),
        )
        .sort("commission_value", descending=True)
    )


def polars_q3_commission_lazy(streaming=False):
    query = (
        pl.scan_parquet(EVENTS_PATH)
        .select(["category_id", "listing_status", "final_price"])
        .join(pl.scan_parquet(DIMENSION_PATH), on="category_id", how="left")
        .filter(pl.col("listing_status") == "sold")
        .with_columns((pl.col("final_price") * pl.col("commission_rate")).alias("commission_value"))
        .group_by("category_group")
        .agg(
            pl.len().alias("sold_listings"),
            pl.col("final_price").sum().alias("gross_value"),
            pl.col("commission_value").sum().alias("commission_value"),
        )
        .sort("commission_value", descending=True)
    )

    if streaming:
        return query.collect(engine="streaming")
    return query.collect()

In [94]:
for query_name, eager_func, lazy_func in [
    ("q1_price_stats_active_listings", polars_q1_price_stats_eager, polars_q1_price_stats_lazy),
    ("q2_top_sellers_by_views", polars_q2_top_sellers_eager, polars_q2_top_sellers_lazy),
    ("q3_commission_by_category_group", polars_q3_commission_eager, polars_q3_commission_lazy),
]:
    benchmark_query(
        "polars",
        "eager",
        query_name,
        eager_func,
        notes="Polars eager read_parquet",
    )

    benchmark_query(
        "polars",
        "lazy",
        query_name,
        lambda lazy_func=lazy_func: lazy_func(streaming=False),
        notes="Polars lazy scan_parquet collect",
    )

    benchmark_query(
        "polars",
        "lazy_streaming",
        query_name,
        lambda lazy_func=lazy_func: lazy_func(streaming=True),
        notes="Polars lazy scan_parquet collect(engine='streaming')",
    )

benchmark_table()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pandas,default,q1_price_stats_active_listings,parquet,default,200000,0.1421,409.3,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas default backend, selected columns Memor..."
1,pandas,pyarrow,q1_price_stats_active_listings,parquet,default,200000,0.0925,422.5,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas PyArrow dtype backend, selected columns..."
2,pandas,default,q2_top_sellers_by_views,parquet,default,200000,0.1645,431.9,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas default backend, selected columns Memor..."
3,pandas,pyarrow,q2_top_sellers_by_views,parquet,default,200000,0.1898,454.0,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas PyArrow dtype backend, selected columns..."
4,pandas,default,q3_commission_by_category_group,parquet,default,200000,0.1680,458.5,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas default backend, selected columns Memor..."
5,pandas,pyarrow,q3_commission_by_category_group,parquet,default,200000,0.1377,461.9,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas PyArrow dtype backend, selected columns..."
6,polars,eager,q1_price_stats_active_listings,parquet,default,200000,0.0241,510.3,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars eager read_parquet Memory is approximat...
7,polars,lazy,q1_price_stats_active_listings,parquet,default,200000,0.0115,519.6,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect Memory is app...
8,polars,lazy_streaming,q1_price_stats_active_listings,parquet,default,200000,0.0147,523.1,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect(engine='strea...
9,polars,eager,q2_top_sellers_by_views,parquet,default,200000,0.0289,580.5,5.45,rows=20;cols=4;checksum=13118326243886092464,Polars eager read_parquet Memory is approximat...


In [95]:
# DuckDB SQL implementations of the three benchmark queries.

def duckdb_q1_price_stats():
    with duckdb.connect() as con:
        return con.execute(
            f"""
            SELECT
                marketplace_category,
                COUNT(*) AS listings,
                AVG(final_price) AS avg_price,
                MEDIAN(final_price) AS median_price,
                QUANTILE_CONT(final_price, 0.90) AS p90_price
            FROM read_parquet('{EVENTS_PATH}')
            WHERE listing_status = 'active'
              AND country IN ('PL', 'DE', 'FR')
              AND event_date >= DATE '2026-03-01'
            GROUP BY marketplace_category
            ORDER BY listings DESC
            """
        ).df()


def duckdb_q2_top_sellers():
    with duckdb.connect() as con:
        return con.execute(
            f"""
            SELECT
                seller_id,
                COUNT(*) AS listings,
                SUM(views_7d) AS total_views,
                AVG(final_price) AS avg_price
            FROM read_parquet('{EVENTS_PATH}')
            GROUP BY seller_id
            ORDER BY total_views DESC, listings DESC
            LIMIT 20
            """
        ).df()


def duckdb_q3_commission():
    with duckdb.connect() as con:
        return con.execute(
            f"""
            SELECT
                d.category_group,
                COUNT(*) AS sold_listings,
                SUM(e.final_price) AS gross_value,
                SUM(e.final_price * d.commission_rate) AS commission_value
            FROM read_parquet('{EVENTS_PATH}') e
            LEFT JOIN read_parquet('{DIMENSION_PATH}') d
              ON e.category_id = d.category_id
            WHERE e.listing_status = 'sold'
            GROUP BY d.category_group
            ORDER BY commission_value DESC
            """
        ).df()

In [96]:
benchmark_query(
    "duckdb",
    "sql",
    "q1_price_stats_active_listings",
    duckdb_q1_price_stats,
    notes="DuckDB SQL over Parquet files",
)

benchmark_query(
    "duckdb",
    "sql",
    "q2_top_sellers_by_views",
    duckdb_q2_top_sellers,
    notes="DuckDB SQL over Parquet files",
)

benchmark_query(
    "duckdb",
    "sql",
    "q3_commission_by_category_group",
    duckdb_q3_commission,
    notes="DuckDB SQL over Parquet files",
)

benchmark_table()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pandas,default,q1_price_stats_active_listings,parquet,default,200000,0.1421,409.3,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas default backend, selected columns Memor..."
1,pandas,pyarrow,q1_price_stats_active_listings,parquet,default,200000,0.0925,422.5,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas PyArrow dtype backend, selected columns..."
2,pandas,default,q2_top_sellers_by_views,parquet,default,200000,0.1645,431.9,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas default backend, selected columns Memor..."
3,pandas,pyarrow,q2_top_sellers_by_views,parquet,default,200000,0.1898,454.0,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas PyArrow dtype backend, selected columns..."
4,pandas,default,q3_commission_by_category_group,parquet,default,200000,0.1680,458.5,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas default backend, selected columns Memor..."
5,pandas,pyarrow,q3_commission_by_category_group,parquet,default,200000,0.1377,461.9,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas PyArrow dtype backend, selected columns..."
6,polars,eager,q1_price_stats_active_listings,parquet,default,200000,0.0241,510.3,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars eager read_parquet Memory is approximat...
7,polars,lazy,q1_price_stats_active_listings,parquet,default,200000,0.0115,519.6,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect Memory is app...
8,polars,lazy_streaming,q1_price_stats_active_listings,parquet,default,200000,0.0147,523.1,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect(engine='strea...
9,polars,eager,q2_top_sellers_by_views,parquet,default,200000,0.0289,580.5,5.45,rows=20;cols=4;checksum=13118326243886092464,Polars eager read_parquet Memory is approximat...


In [97]:
# PySpark implementations of the three benchmark queries.

from pyspark.sql import functions as F


def spark_q1_price_stats():
    df = spark.read.parquet(str(EVENTS_PATH)).select(
        "event_date",
        "country",
        "listing_status",
        "marketplace_category",
        "final_price",
    )

    result = (
        df.filter(
            (F.col("listing_status") == "active")
            & (F.col("country").isin("PL", "DE", "FR"))
            & (F.col("event_date") >= F.lit("2026-03-01"))
        )
        .groupBy("marketplace_category")
        .agg(
            F.count("*").alias("listings"),
            F.avg("final_price").alias("avg_price"),
            F.expr("percentile_approx(final_price, 0.5)").alias("median_price"),
            F.expr("percentile_approx(final_price, 0.9)").alias("p90_price"),
        )
        .orderBy(F.desc("listings"))
    )

    return result.toPandas()


def spark_q2_top_sellers():
    df = spark.read.parquet(str(EVENTS_PATH)).select(
        "seller_id",
        "views_7d",
        "final_price",
    )

    result = (
        df.groupBy("seller_id")
        .agg(
            F.count("*").alias("listings"),
            F.sum("views_7d").alias("total_views"),
            F.avg("final_price").alias("avg_price"),
        )
        .orderBy(F.desc("total_views"), F.desc("listings"))
        .limit(20)
    )

    return result.toPandas()


def spark_q3_commission():
    events_df = spark.read.parquet(str(EVENTS_PATH)).select(
        "category_id",
        "listing_status",
        "final_price",
    )

    dimension_df = spark.read.parquet(str(DIMENSION_PATH))

    result = (
        events_df.join(dimension_df, on="category_id", how="left")
        .filter(F.col("listing_status") == "sold")
        .withColumn("commission_value", F.col("final_price") * F.col("commission_rate"))
        .groupBy("category_group")
        .agg(
            F.count("*").alias("sold_listings"),
            F.sum("final_price").alias("gross_value"),
            F.sum("commission_value").alias("commission_value"),
        )
        .orderBy(F.desc("commission_value"))
    )

    return result.toPandas()

In [98]:
benchmark_query(
    "pyspark",
    "local[*]",
    "q1_price_stats_active_listings",
    spark_q1_price_stats,
    notes="PySpark local mode, results collected to Pandas",
)

benchmark_query(
    "pyspark",
    "local[*]",
    "q2_top_sellers_by_views",
    spark_q2_top_sellers,
    notes="PySpark local mode, results collected to Pandas",
)

benchmark_query(
    "pyspark",
    "local[*]",
    "q3_commission_by_category_group",
    spark_q3_commission,
    notes="PySpark local mode, results collected to Pandas",
)

benchmark_table()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pandas,default,q1_price_stats_active_listings,parquet,default,200000,0.1421,409.3,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas default backend, selected columns Memor..."
1,pandas,pyarrow,q1_price_stats_active_listings,parquet,default,200000,0.0925,422.5,5.45,rows=6;cols=5;checksum=17610955209037473064,"Pandas PyArrow dtype backend, selected columns..."
2,pandas,default,q2_top_sellers_by_views,parquet,default,200000,0.1645,431.9,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas default backend, selected columns Memor..."
3,pandas,pyarrow,q2_top_sellers_by_views,parquet,default,200000,0.1898,454.0,5.45,rows=20;cols=4;checksum=13118326243886092464,"Pandas PyArrow dtype backend, selected columns..."
4,pandas,default,q3_commission_by_category_group,parquet,default,200000,0.1680,458.5,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas default backend, selected columns Memor..."
5,pandas,pyarrow,q3_commission_by_category_group,parquet,default,200000,0.1377,461.9,5.45,rows=4;cols=4;checksum=18376955833962582745,"Pandas PyArrow dtype backend, selected columns..."
6,polars,eager,q1_price_stats_active_listings,parquet,default,200000,0.0241,510.3,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars eager read_parquet Memory is approximat...
7,polars,lazy,q1_price_stats_active_listings,parquet,default,200000,0.0115,519.6,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect Memory is app...
8,polars,lazy_streaming,q1_price_stats_active_listings,parquet,default,200000,0.0147,523.1,5.45,rows=6;cols=5;checksum=17610955209037473064,Polars lazy scan_parquet collect(engine='strea...
9,polars,eager,q2_top_sellers_by_views,parquet,default,200000,0.0289,580.5,5.45,rows=20;cols=4;checksum=13118326243886092464,Polars eager read_parquet Memory is approximat...


### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [99]:
# TODO 2.5: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

import duckdb
import polars as pl


events_df = pl.read_parquet(EVENTS_PATH)

events_df.sort(["event_date", "listing_status"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

csv_columns = ["event_date", "country", "listing_status", "marketplace_category", "final_price"]
events_df.select(csv_columns).write_csv(CSV_EVENTS_PATH)

def duckdb_q1_custom(path, is_csv=False):
    with duckdb.connect() as con:
        read_expr = f"read_csv_auto('{path}')" if is_csv else f"read_parquet('{path}')"
        return con.execute(
            f"""
            SELECT
                marketplace_category,
                COUNT(*) AS listings,
                AVG(final_price) AS avg_price,
                MEDIAN(final_price) AS median_price,
                QUANTILE_CONT(final_price, 0.90) AS p90_price
            FROM {read_expr}
            WHERE listing_status = 'active'
              AND country IN ('PL', 'DE', 'FR')
              AND event_date >= DATE '2026-03-01'
            GROUP BY marketplace_category
            ORDER BY listings DESC
            """
        ).df()

benchmark_query(
    "duckdb", "layout_default_parquet", "q1_price_stats",
    lambda: duckdb_q1_custom(EVENTS_PATH),
    input_path=EVENTS_PATH, data_format="parquet", layout="default",
    notes="Random order Parquet"
)

benchmark_query(
    "duckdb", "layout_optimized_parquet", "q1_price_stats",
    lambda: duckdb_q1_custom(OPTIMIZED_EVENTS_PATH),
    input_path=OPTIMIZED_EVENTS_PATH, data_format="parquet", layout="sorted_date_status_rg100k",
    notes="Sorted by event_date & listing_status, rg_size=100k"
)

benchmark_query(
    "duckdb", "layout_csv", "q1_price_stats",
    lambda: duckdb_q1_custom(CSV_EVENTS_PATH, is_csv=True),
    input_path=CSV_EVENTS_PATH, data_format="csv", layout="flat_subset",
    notes="CSV baseline (no pushdown)"
)

display(benchmark_table().tail(3))



,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
21,duckdb,layout_default_parquet,q1_price_stats,parquet,default,200000,0.0553,700.4,5.45,rows=6;cols=5;checksum=17610955209037473064,Random order Parquet Memory is approximate RSS...
22,duckdb,layout_optimized_parquet,q1_price_stats,parquet,sorted_date_status_rg100k,200000,0.0515,701.3,5.39,rows=6;cols=5;checksum=17610955209037473064,"Sorted by event_date & listing_status, rg_size..."
23,duckdb,layout_csv,q1_price_stats,csv,flat_subset,200000,0.3439,704.2,6.90,rows=6;cols=5;checksum=17610955209037473064,CSV baseline (no pushdown) Memory is approxima...


### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [100]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

import polars as pl

# Ścieżka wyjściowa dla eksperymentu z sink_parquet
SINK_PATH = OUTPUT_DIR / "polars_sink_output.parquet"

def q_transform_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return (
        df.filter((pl.col("listing_status") == "active") & (pl.col("country").is_in(["PL", "DE", "FR"])))
        .with_columns((pl.col("price") * 1.23).alias("price_with_tax"))
    )

def q_transform_lazy():
    return (
        pl.scan_parquet(EVENTS_PATH)
        .filter((pl.col("listing_status") == "active") & (pl.col("country").is_in(["PL", "DE", "FR"])))
        .with_columns((pl.col("price") * 1.23).alias("price_with_tax"))
        .collect()
    )

def q_transform_streaming_collect():
    return (
        pl.scan_parquet(EVENTS_PATH)
        .filter((pl.col("listing_status") == "active") & (pl.col("country").is_in(["PL", "DE", "FR"])))
        .with_columns((pl.col("price") * 1.23).alias("price_with_tax"))
        .collect(engine="streaming")
    )

def q_transform_streaming_sink():
    (
        pl.scan_parquet(EVENTS_PATH)
        .filter((pl.col("listing_status") == "active") & (pl.col("country").is_in(["PL", "DE", "FR"])))
        .with_columns((pl.col("price") * 1.23).alias("price_with_tax"))
        .sink_parquet(SINK_PATH)
    )
    return pl.read_parquet(SINK_PATH)

benchmark_query(
    "polars", "eager", "q_transform_large",
    q_transform_eager,
    input_path=EVENTS_PATH, data_format="parquet", layout="default",
    notes="Eager: read_parquet -> filter -> transform"
)

benchmark_query(
    "polars", "lazy", "q_transform_large",
    q_transform_lazy,
    input_path=EVENTS_PATH, data_format="parquet", layout="default",
    notes="Lazy: scan_parquet -> collect()"
)

benchmark_query(
    "polars", "lazy_streaming", "q_transform_large",
    q_transform_streaming_collect,
    input_path=EVENTS_PATH, data_format="parquet", layout="default",
    notes="Streaming: scan_parquet -> collect(engine='streaming')"
)

benchmark_query(
    "polars", "lazy_sink", "q_transform_large",
    q_transform_streaming_sink,
    input_path=EVENTS_PATH, data_format="parquet", layout="default",
    notes="Sink: scan_parquet -> sink_parquet()"
)

display(benchmark_table().tail(4))



,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
24,polars,eager,q_transform_large,parquet,default,200000,0.0733,844.5,5.45,rows=55612;cols=19;checksum=13341463155652266660,Eager: read_parquet -> filter -> transform Mem...
25,polars,lazy,q_transform_large,parquet,default,200000,0.0542,884.6,5.45,rows=55612;cols=19;checksum=13341463155652266660,Lazy: scan_parquet -> collect() Memory is appr...
26,polars,lazy_streaming,q_transform_large,parquet,default,200000,0.0883,876.4,5.45,rows=55612;cols=19;checksum=13341463155652266660,Streaming: scan_parquet -> collect(engine='str...
27,polars,lazy_sink,q_transform_large,parquet,default,200000,0.1604,886.0,5.45,rows=55612;cols=19;checksum=13341463155652266660,Sink: scan_parquet -> sink_parquet() Memory is...


#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [101]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
Polars struggles with workloads that exceed single-node limits, specifically when 
severe data skew in `group-by` or `join` operations causes intermediate data to exceed 
local RAM and disk capacity. Furthermore, it lacks distributed fault tolerance; if a 
local resource is exhausted, the entire query fails.
"""
POLARS_LIMITATION_EVIDENCE = """
As observed in Task 3.1, while Polars' `streaming` engine and `sink_parquet` drastically 
reduce memory footprints for row-wise transformations, global operations like high-cardinality 
joins or skewed aggregations still require significant local memory/disk. If the dataset scales to 
hundreds of millions of rows, the single-node architecture becomes a bottleneck. Spark handles this by 
shuffling data across a cluster and recomputing failed partitions, whereas Polars would simply crash with 
an Out-Of-Memory (OOM) error.
"""

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


**Polars limitation scenario**

Polars struggles with workloads that exceed single-node limits, specifically when 
severe data skew in `group-by` or `join` operations causes intermediate data to exceed 
local RAM and disk capacity. Furthermore, it lacks distributed fault tolerance; if a 
local resource is exhausted, the entire query fails.

**Evidence**

As observed in Task 3.1, while Polars' `streaming` engine and `sink_parquet` drastically 
reduce memory footprints for row-wise transformations, global operations like high-cardinality 
joins or skewed aggregations still require significant local memory/disk. If the dataset scales to 
hundreds of millions of rows, the single-node architecture becomes a bottleneck. Spark handles this by 
shuffling data across a cluster and recomputing failed partitions, whereas Polars would simply crash with 
an Out-Of-Memory (OOM) error.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [102]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
Switch to Spark when the dataset exceeds single-machine RAM and disk limits, 
or when the workload requires distributed fault tolerance and cluster scaling.
"""

DECISION_EVIDENCE = """
Our benchmarks show Polars and DuckDB easily out-perform local Spark on data 
that fits locally. However, if Polars exhausts local memory during a heavy join, 
the query crashes (OOM), whereas Spark distributes the load across executors to prevent failure.
"""


display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Switch to Spark when the dataset exceeds single-machine RAM and disk limits, 
or when the workload requires distributed fault tolerance and cluster scaling.

**Evidence**

Our benchmarks show Polars and DuckDB easily out-perform local Spark on data 
that fits locally. However, if Polars exhausts local memory during a heavy join, 
the query crashes (OOM), whereas Spark distributes the load across executors to prevent failure.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [103]:
# TODO: Run selected scalability experiments and append results to benchmark_results.
import duckdb
from pyspark.sql import SparkSession

print("Running scalability experiments (DuckDB and PySpark)...")

# --- 1. DuckDB Scalability ---
def duckdb_scale_test(threads):
    with duckdb.connect() as con:
        con.execute(f"PRAGMA threads={threads}")
        con.execute(f"""
            SELECT marketplace_category, COUNT(*) 
            FROM read_parquet('{EVENTS_PATH}') 
            GROUP BY marketplace_category
        """).df()

benchmark_query("duckdb", "threads=1", "q_scale", lambda: duckdb_scale_test(1), notes="1 thread")
benchmark_query("duckdb", "threads=2", "q_scale", lambda: duckdb_scale_test(2), notes="2 threads")
benchmark_query("duckdb", "threads=4", "q_scale", lambda: duckdb_scale_test(4), notes="4 threads")

# --- 2. PySpark Scalability ---
def spark_scale_test(cores):
    # Restarts Spark context with a new core limit
    global spark
    if 'spark' in globals() and spark is not None:
        spark.stop()
    
    spark = SparkSession.builder.master(f"local[{cores}]").appName("Scale").getOrCreate()
    spark.read.parquet(str(EVENTS_PATH)).groupBy("marketplace_category").count().collect()

benchmark_query("pyspark", "local[1]", "q_scale", lambda: spark_scale_test(1), notes="1 core")
benchmark_query("pyspark", "local[2]", "q_scale", lambda: spark_scale_test(2), notes="2 cores")
benchmark_query("pyspark", "local[*]", "q_scale", lambda: spark_scale_test("*"), notes="All available cores")

display(benchmark_table().tail(6))


Running scalability experiments (DuckDB and PySpark)...


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
28,duckdb,threads=1,q_scale,parquet,default,200000,0.0540,908.2,5.45,rows=0;cols=0;checksum=0,1 thread Memory is approximate RSS before/afte...
29,duckdb,threads=2,q_scale,parquet,default,200000,0.0703,908.3,5.45,rows=0;cols=0;checksum=0,2 threads Memory is approximate RSS before/aft...
30,duckdb,threads=4,q_scale,parquet,default,200000,0.0398,909.2,5.45,rows=0;cols=0;checksum=0,4 threads Memory is approximate RSS before/aft...
31,pyspark,local[1],q_scale,parquet,default,200000,1.1410,909.7,5.45,rows=0;cols=0;checksum=0,1 core Memory is approximate RSS before/after ...
32,pyspark,local[2],q_scale,parquet,default,200000,1.1831,909.7,5.45,rows=0;cols=0;checksum=0,2 cores Memory is approximate RSS before/after...
33,pyspark,local[*],q_scale,parquet,default,200000,1.3052,869.6,5.45,rows=0;cols=0;checksum=0,All available cores Memory is approximate RSS ...


## RESULTS Task 4: Scalability Analysis

As seen in the results, doubling the number of cores does not exactly halve the execution time (scaling is not perfectly linear). This happens due to several factors:

1. **I/O Bottleneck**: CPU cores process data much faster than it can be read from the disk. When adding more threads, they often sit idle waiting for disk I/O.
2. **Synchronization Overhead**: Distributing tasks and merging intermediate results across multiple threads introduces locking and scheduling overhead (especially visible in Spark JVM).
3. **Amdahl's Law**: Some parts of the query (like the final reduction/aggregation step) must be executed sequentially by a single thread, which sets a hard limit on the maximum possible speedup.


### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [104]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

import subprocess
import importlib
import shutil

PROJECT_ID = "tbd-2026l-318786"
DATAPROC_REGION = "europe-west1"
DATAPROC_CLUSTER = "tbd-cluster"
GCS_DATA_BUCKET = "gs://tbd-2026l-318786-data"

TASK5_DIR = Path("task5_dataproc")
TASK5_DIR.mkdir(exist_ok=True)

GCS_PREFIX = f"{GCS_DATA_BUCKET}/phase2_26L/group_{GROUP_ID:02d}/{SCALE}"

GCS_EVENTS_PATH = f"{GCS_PREFIX}/events.parquet"
GCS_PARTITIONED_EVENTS_DIR = f"{GCS_PREFIX}/events_partitioned"
GCS_DIMENSION_PATH = f"{GCS_PREFIX}/dimension.parquet"

GCS_JOB_SCRIPT = f"{GCS_PREFIX}/jobs/task5_dataproc_job.py"
GCS_RESULTS_DIR = f"{GCS_PREFIX}/dataproc_results/run_{int(time.time())}"

LOCAL_JOB_SCRIPT_PATH = TASK5_DIR / "task5_dataproc_job.py"
LOCAL_RUNNER_PATH = TASK5_DIR / "task5_dataproc_runner.py"
LOCAL_RESULTS_DIR = TASK5_DIR / "results"

print("PROJECT_ID:", PROJECT_ID)
print("DATAPROC_REGION:", DATAPROC_REGION)
print("DATAPROC_CLUSTER:", DATAPROC_CLUSTER)
print("GCS_PREFIX:", GCS_PREFIX)
print("GCS_RESULTS_DIR:", GCS_RESULTS_DIR)
print("N_ROWS:", N_ROWS)

PROJECT_ID: tbd-2026l-318786
DATAPROC_REGION: europe-west1
DATAPROC_CLUSTER: tbd-cluster
GCS_PREFIX: gs://tbd-2026l-318786-data/phase2_26L/group_14/debug
GCS_RESULTS_DIR: gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dataproc_results/run_1781120582
N_ROWS: 200000


In [105]:
def local_path_size_mb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / (1024 ** 2)
    if path.is_dir():
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024 ** 2)
    return 0.0

events_size_mb = path_size_mb(EVENTS_PATH) if "path_size_mb" in globals() else local_path_size_mb(EVENTS_PATH)
dimension_size_mb = path_size_mb(DIMENSION_PATH) if "path_size_mb" in globals() else local_path_size_mb(DIMENSION_PATH)

if Path(PARTITIONED_EVENTS_DIR).exists():
    partitioned_size_mb = (
        path_size_mb(PARTITIONED_EVENTS_DIR)
        if "path_size_mb" in globals()
        else local_path_size_mb(PARTITIONED_EVENTS_DIR)
    )
else:
    partitioned_size_mb = events_size_mb

print("EVENTS_PATH:", EVENTS_PATH, "| size MB:", round(events_size_mb, 2))
print("DIMENSION_PATH:", DIMENSION_PATH, "| size MB:", round(dimension_size_mb, 2))
print("PARTITIONED_EVENTS_DIR:", PARTITIONED_EVENTS_DIR, "| size MB:", round(partitioned_size_mb, 2))

EVENTS_PATH: ..\data\phase2_26L\group_14\events.parquet | size MB: 5.45
DIMENSION_PATH: ..\data\phase2_26L\group_14\dimension.parquet | size MB: 0.0
PARTITIONED_EVENTS_DIR: ..\data\phase2_26L\group_14\events_partitioned | size MB: 5.93


In [ ]:
if str(TASK5_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR.resolve()))

import task5_dataproc_runner
importlib.reload(task5_dataproc_runner)

print("Using gcloud:", task5_dataproc_runner.GCLOUD)

available_clusters = task5_dataproc_runner.check_dataproc_cluster(
    project_id=PROJECT_ID,
    region=DATAPROC_REGION,
    cluster_name=DATAPROC_CLUSTER,
)

print("Available Dataproc clusters:", available_clusters)

## Local run

In [107]:
from pyspark.sql import SparkSession

if "spark" not in globals() or spark is None:
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("TBDPhase2LocalSparkBaseline")
        .getOrCreate()
    )

local_spark_query_funcs = {
    "q1_price_stats_active_listings": spark_q1_price_stats,
    "q2_top_sellers_by_views": spark_q2_top_sellers,
    "q3_commission_by_category_group": spark_q3_commission,
}

current_results = (
    benchmark_table()
    if len(benchmark_results) > 0
    else pd.DataFrame(columns=BENCHMARK_COLUMNS)
)

for query_name, query_func in local_spark_query_funcs.items():
    already_exists = (
        len(current_results) > 0
        and (
            (current_results["library_engine"] == "pyspark")
            & (current_results["mode"] == "local[*]")
            & (current_results["query_name"] == query_name)
        ).any()
    )

    if not already_exists:
        benchmark_query(
            "pyspark",
            "local[*]",
            query_name,
            query_func,
            input_path=EVENTS_PATH,
            data_format="parquet",
            layout="default",
            notes="PySpark local baseline for Dataproc comparison.",
        )

display(
    benchmark_table()[
        (benchmark_table()["library_engine"] == "pyspark")
        & (benchmark_table()["query_name"].isin(local_spark_query_funcs.keys()))
    ]
)

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
18,pyspark,local[*],q1_price_stats_active_listings,parquet,default,200000,1.3812,573.7,5.45,rows=6;cols=5;checksum=2440908405210733411,"PySpark local mode, results collected to Panda..."
19,pyspark,local[*],q2_top_sellers_by_views,parquet,default,200000,0.6586,556.8,5.45,rows=20;cols=4;checksum=13118326243886092464,"PySpark local mode, results collected to Panda..."
20,pyspark,local[*],q3_commission_by_category_group,parquet,default,200000,1.0439,554.3,5.45,rows=4;cols=4;checksum=18376955833962582745,"PySpark local mode, results collected to Panda..."


## Dataproc run

In [108]:
dataproc_q1_input_path, dataproc_q1_layout = task5_dataproc_runner.upload_task5_inputs(
    events_path=EVENTS_PATH,
    dimension_path=DIMENSION_PATH,
    partitioned_events_dir=PARTITIONED_EVENTS_DIR,
    gcs_events_path=GCS_EVENTS_PATH,
    gcs_dimension_path=GCS_DIMENSION_PATH,
    gcs_partitioned_events_dir=GCS_PARTITIONED_EVENTS_DIR,
)

task5_dataproc_runner.upload_job_script(
    local_job_script_path=LOCAL_JOB_SCRIPT_PATH,
    gcs_job_script=GCS_JOB_SCRIPT,
)

print("GCS_EVENTS_PATH:", GCS_EVENTS_PATH)
print("GCS_DIMENSION_PATH:", GCS_DIMENSION_PATH)
print("Dataproc q1 input path:", dataproc_q1_input_path)
print("Dataproc q1 layout:", dataproc_q1_layout)
print("GCS_JOB_SCRIPT:", GCS_JOB_SCRIPT)

+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage cp ..\data\phase2_26L\group_14\events.parquet gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/events.parquet
+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage cp ..\data\phase2_26L\group_14\dimension.parquet gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dimension.parquet
+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage rsync -r ..\data\phase2_26L\group_14\events_partitioned gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/events_partitioned
+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage cp task5_dataproc\task5_dataproc_job.py gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/jobs/task5_dataproc_job.py
GCS_EVENTS_PATH: gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/events.parquet
GCS_DIMENSION_PATH: gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dimensio

## Dataproc submit job

In [109]:
dataproc_wall_clock_s = task5_dataproc_runner.submit_dataproc_job(
    project_id=PROJECT_ID,
    region=DATAPROC_REGION,
    cluster_name=DATAPROC_CLUSTER,
    gcs_job_script=GCS_JOB_SCRIPT,
    gcs_events_path=GCS_EVENTS_PATH,
    gcs_partitioned_or_default_path=dataproc_q1_input_path,
    gcs_dimension_path=GCS_DIMENSION_PATH,
    gcs_results_dir=GCS_RESULTS_DIR,
    rows=N_ROWS,
    events_size_mb=events_size_mb,
    partitioned_size_mb=partitioned_size_mb,
    repeats=3,
)

print("Dataproc wall-clock time including job submission:", dataproc_wall_clock_s, "s")
print("Results path:", GCS_RESULTS_DIR)

+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd dataproc jobs submit pyspark gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/jobs/task5_dataproc_job.py --project tbd-2026l-318786 --region europe-west1 --cluster tbd-cluster --properties spark.sql.shuffle.partitions=8 -- --events-path gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/events.parquet --partitioned-events-path gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/events_partitioned --dimension-path gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dimension.parquet --output-path gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dataproc_results/run_1781120582 --rows 200000 --events-size-mb 5.448214530944824 --partitioned-size-mb 5.926347732543945 --repeats 3
Dataproc wall-clock time including job submission: 140.26 s
Results path: gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dataproc_results/run_1781120582


## Dataproc download results

In [110]:
dataproc_rows = task5_dataproc_runner.download_dataproc_results(
    gcs_results_dir=GCS_RESULTS_DIR,
    local_results_dir=LOCAL_RESULTS_DIR,
)

for row in dataproc_rows:
    row["notes"] += (
        f" Dataproc wall-clock including job submission: "
        f"{dataproc_wall_clock_s}s."
    )

task5_dataproc_results = pd.DataFrame(dataproc_rows, columns=BENCHMARK_COLUMNS)

existing_keys = {
    (
        row["library_engine"],
        row["mode"],
        row["query_name"],
        row["layout"],
        row["rows"],
        row["median_time_s"],
    )
    for row in benchmark_results
}

for row in task5_dataproc_results.to_dict("records"):
    key = (
        row["library_engine"],
        row["mode"],
        row["query_name"],
        row["layout"],
        row["rows"],
        row["median_time_s"],
    )

    if key not in existing_keys:
        benchmark_results.append(row)

display(task5_dataproc_results)

+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage ls gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dataproc_results/run_1781120582/part-*
+ C:\Users\jakub\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd storage cp gs://tbd-2026l-318786-data/phase2_26L/group_14/debug/dataproc_results/run_1781120582/part-00000 task5_dataproc\results


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pyspark,dataproc-cluster,q1_price_stats_active_listings,parquet,gcs_partitioned_by_event_date,200000,9.8613,-1.0,5.93,rows=6;cols=5;checksum=1fc8eff1308ce3e5,Dataproc PySpark. Query uses date filter over ...
1,pyspark,dataproc-cluster,q2_top_sellers_by_views,parquet,gcs_default,200000,4.1142,-1.0,5.45,rows=20;cols=4;checksum=97e6f6000f668e03,Dataproc PySpark. High-cardinality groupBy ove...
2,pyspark,dataproc-cluster,q3_commission_by_category_group,parquet,gcs_default,200000,4.1148,-1.0,5.45,rows=4;cols=4;checksum=c352b7c3c4468476,Dataproc PySpark. Join with dimension table fo...


## Task 5 - results

In [111]:
task5_query_names = [
    "q1_price_stats_active_listings",
    "q2_top_sellers_by_views",
    "q3_commission_by_category_group",
]

all_results = benchmark_table()

task5_spark_comparison = (
    all_results[
        (all_results["library_engine"] == "pyspark")
        & (all_results["query_name"].isin(task5_query_names))
        & (all_results["mode"].isin(["local[*]", "dataproc-cluster"]))
    ]
    .sort_values(["query_name", "mode", "median_time_s"])
    .drop_duplicates(["query_name", "mode"], keep="last")
    .reset_index(drop=True)
)

display(
    task5_spark_comparison[
        [
            "mode",
            "query_name",
            "layout",
            "rows",
            "median_time_s",
            "peak_memory_mb",
            "input_size_mb",
            "result_check",
            "notes",
        ]
    ]
)

task5_spark_comparison_wide = (
    task5_spark_comparison
    .pivot_table(
        index="query_name",
        columns="mode",
        values="median_time_s",
        aggfunc="last",
    )
    .reset_index()
)

if {"local[*]", "dataproc-cluster"}.issubset(task5_spark_comparison_wide.columns):
    task5_spark_comparison_wide["dataproc_vs_local_ratio"] = (
        task5_spark_comparison_wide["dataproc-cluster"]
        / task5_spark_comparison_wide["local[*]"]
    ).round(3)

display(task5_spark_comparison_wide)

,mode,query_name,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,dataproc-cluster,q1_price_stats_active_listings,gcs_partitioned_by_event_date,200000,9.8613,-1.0,5.93,rows=6;cols=5;checksum=1fc8eff1308ce3e5,Dataproc PySpark. Query uses date filter over ...
1,local[*],q1_price_stats_active_listings,default,200000,1.3812,573.7,5.45,rows=6;cols=5;checksum=2440908405210733411,"PySpark local mode, results collected to Panda..."
2,dataproc-cluster,q2_top_sellers_by_views,gcs_default,200000,4.1142,-1.0,5.45,rows=20;cols=4;checksum=97e6f6000f668e03,Dataproc PySpark. High-cardinality groupBy ove...
3,local[*],q2_top_sellers_by_views,default,200000,0.6586,556.8,5.45,rows=20;cols=4;checksum=13118326243886092464,"PySpark local mode, results collected to Panda..."
4,dataproc-cluster,q3_commission_by_category_group,gcs_default,200000,4.1148,-1.0,5.45,rows=4;cols=4;checksum=c352b7c3c4468476,Dataproc PySpark. Join with dimension table fo...
5,local[*],q3_commission_by_category_group,default,200000,1.0439,554.3,5.45,rows=4;cols=4;checksum=18376955833962582745,"PySpark local mode, results collected to Panda..."


mode,query_name,dataproc-cluster,local[*],dataproc_vs_local_ratio
0,q1_price_stats_active_listings,9.8613,1.3812,7.140
1,q2_top_sellers_by_views,4.1142,0.6586,6.247
2,q3_commission_by_category_group,4.1148,1.0439,3.942


In [11]:
DATAPROC_EVIDENCE = """
The Dataproc experiment compared local PySpark with PySpark submitted to the
`tbd-cluster` Dataproc cluster on the same generated Parquet data uploaded to GCS.
For this 200000-row, 5.45 MB dataset, Dataproc was slower than local Spark because
the fixed overhead of job submission, cluster scheduling, GCS reads, and distributed
task coordination dominated the actual query execution.

The most relevant Spark execution characteristic was visible in q2 and q3. Query q2
performs a high-cardinality groupBy over seller_id, which requires shuffle. Query q3
performs a join followed by aggregation, which also requires distributed planning and
shuffle work. Query q1 used the partitioned GCS layout by event_date, which is the
appropriate layout when filtering by date because Spark can avoid reading irrelevant
partitions.
"""
display_answer("Dataproc evidence", DATAPROC_EVIDENCE)

**Dataproc evidence**

The Dataproc experiment compared local PySpark with PySpark submitted to the
`tbd-cluster` Dataproc cluster on the same generated Parquet data uploaded to GCS.
For this 200000-row, 5.45 MB dataset, Dataproc was slower than local Spark because
the fixed overhead of job submission, cluster scheduling, GCS reads, and distributed
task coordination dominated the actual query execution.

The most relevant Spark execution characteristic was visible in q2 and q3. Query q2
performs a high-cardinality groupBy over seller_id, which requires shuffle. Query q3
performs a join followed by aggregation, which also requires distributed planning and
shuffle work. Query q1 used the partitioned GCS layout by event_date, which is the
appropriate layout when filtering by date because Spark can avoid reading irrelevant
partitions.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [3]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
The query that best exposes the difference between DataFrame and SQL engines is
`q3_commission_by_category_group`.

This query is naturally SQL-shaped because it combines a join with a dimension table,
a filter on sold listings, a computed commission column, and a grouped aggregation.
In the benchmark, DuckDB SQL executed this query in about 0.0667s, while Pandas 
default took about 0.1680s and local PySpark took about 1.0439s. Polars lazy was still the
fastest at about 0.0229s, but the query clearly shows how an SQL engine can optimize
a declarative join + aggregation plan without manual DataFrame steps such as merge,
copying, and explicit intermediate materialization.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

The query that best exposes the difference between DataFrame and SQL engines is
`q3_commission_by_category_group`.

This query is naturally SQL-shaped because it combines a join with a dimension table,
a filter on sold listings, a computed commission column, and a grouped aggregation.
In the benchmark, DuckDB SQL executed this query in about 0.0667s, while Pandas 
default took about 0.1680s and local PySpark took about 1.0439s. Polars lazy was still the
fastest at about 0.0229s, but the query clearly shows how an SQL engine can optimize
a declarative join + aggregation plan without manual DataFrame steps such as merge,
copying, and explicit intermediate materialization.

In [4]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
The most memory-sensitive measured workload was `q_transform_large` from the Polars
execution-mode experiment, because it returned a large result:
55612 rows and 19 columns. Its measured peak RSS was around 844.5 MB for eager,
884.6 MB for lazy, 876.4 MB for lazy streaming, and 886.0 MB for sink_parquet.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

The most memory-sensitive measured workload was `q_transform_large` from the Polars
execution-mode experiment, because it returned a large result:
55612 rows and 19 columns. Its measured peak RSS was around 844.5 MB for eager,
884.6 MB for lazy, 876.4 MB for lazy streaming, and 886.0 MB for sink_parquet.

In [5]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
Lazy execution changed when and how data was materialized. 
With Polars lazy execution, the query plan could push column selection
and filters into the Parquet scan, so unnecessary columns and intermediate DataFrames
did not have to be materialized eagerly in Python.

This is visible in the timings: for q1, Polars eager took about 0.0241s while lazy took
about 0.0115s; for q2, eager took about 0.0289s while lazy took about 0.0215s; and 
for q3, eager took about 0.0492s while lazy took about 0.0229s. The reported input file size
remained the same, about 5.45 MB, because the benchmark records the physical file size,
not the exact number of bytes actually scanned after projection or predicate pushdown.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

Lazy execution changed when and how data was materialized. 
With Polars lazy execution, the query plan could push column selection
and filters into the Parquet scan, so unnecessary columns and intermediate DataFrames
did not have to be materialized eagerly in Python.

This is visible in the timings: for q1, Polars eager took about 0.0241s while lazy took
about 0.0115s; for q2, eager took about 0.0289s while lazy took about 0.0215s; and 
for q3, eager took about 0.0492s while lazy took about 0.0229s. The reported input file size
remained the same, about 5.45 MB, because the benchmark records the physical file size,
not the exact number of bytes actually scanned after projection or predicate pushdown.

In [6]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
streaming collection did not clearly reduce either memory or runtime.
For `q_transform_large`, normal lazy collection took about 0.0542s with around 884.6 MB
peak RSS, while `collect(engine="streaming")` took about 0.0883s with around 876.4 MB
peak RSS. That means streaming collection was slightly lower in measured memory, but
slower in runtime, and the memory difference was small.

The important distinction is that `collect(engine="streaming")` still returns a DataFrame
to Python, so the final result is still materialized in memory. In contrast,
`sink_parquet(...)` writes the result directly to disk.
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

streaming collection did not clearly reduce either memory or runtime.
For `q_transform_large`, normal lazy collection took about 0.0542s with around 884.6 MB
peak RSS, while `collect(engine="streaming")` took about 0.0883s with around 876.4 MB
peak RSS. That means streaming collection was slightly lower in measured memory, but
slower in runtime, and the memory difference was small.

The important distinction is that `collect(engine="streaming")` still returns a DataFrame
to Python, so the final result is still materialized in memory. In contrast,
`sink_parquet(...)` writes the result directly to disk.

In [7]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
A streaming sink is more appropriate when the query output is large and the final result
does not need to be materialized as a Python DataFrame. In this notebook,
`q_transform_large` produced 55612 rows and 19 columns, which is larger than the small
aggregate outputs from q1, q2, and q3, but it was still small enough to collect locally.

For the current dataset size, collecting the result was acceptable and faster. However,
if the same transformation produced millions of rows or a file intended for later
processing, `sink_parquet(...)` would be the better choice because it writes directly
to disk and avoids keeping the full output in Python memory.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

A streaming sink is more appropriate when the query output is large and the final result
does not need to be materialized as a Python DataFrame. In this notebook,
`q_transform_large` produced 55612 rows and 19 columns, which is larger than the small
aggregate outputs from q1, q2, and q3, but it was still small enough to collect locally.

For the current dataset size, collecting the result was acceptable and faster. However,
if the same transformation produced millions of rows or a file intended for later
processing, `sink_parquet(...)` would be the better choice because it writes directly
to disk and avoids keeping the full output in Python memory.

In [8]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
Yes, local Spark behaved as expected for this dataset size. The main dataset had only
200000 rows and about 5.45 MB, so single-node engines had much less overhead than Spark.

Local PySpark was slower than Pandas, Polars, and DuckDB on all three benchmark queries:
q1 took about 1.3812s in local Spark, q2 about 0.6586s, and q3 about 1.0439s. In comparison,
Polars lazy completed the same queries in about 0.0115s, 0.0215s, and 0.0229s, while
DuckDB SQL completed them in about 0.0673s, 0.1332s, and 0.0667s.

This is expected because Spark has JVM startup cost, task scheduling overhead, DataFrame
planning overhead, and shuffle overhead for group-by/join operations.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

Yes, local Spark behaved as expected for this dataset size. The main dataset had only
200000 rows and about 5.45 MB, so single-node engines had much less overhead than Spark.

Local PySpark was slower than Pandas, Polars, and DuckDB on all three benchmark queries:
q1 took about 1.3812s in local Spark, q2 about 0.6586s, and q3 about 1.0439s. In comparison,
Polars lazy completed the same queries in about 0.0115s, 0.0215s, and 0.0229s, while
DuckDB SQL completed them in about 0.0673s, 0.1332s, and 0.0667s.

This is expected because Spark has JVM startup cost, task scheduling overhead, DataFrame
planning overhead, and shuffle overhead for group-by/join operations.

In [9]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
I would stay local as ong as the dataset and intermediate
results comfortably fit in memory and the workload finishes quickly on one machine.
I would move to a cluster when the input becomes at least tens of GB, when intermediate
shuffle/join/group-by state approaches local RAM limits, or when the query must process
data that cannot be safely materialized on a single machine. Query shape matters too:
high-cardinality group-bys, large joins, and large output transformations are stronger
reasons to move to Spark/Dataproc than small aggregations returning only a few rows.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

I would stay local as ong as the dataset and intermediate
results comfortably fit in memory and the workload finishes quickly on one machine.
I would move to a cluster when the input becomes at least tens of GB, when intermediate
shuffle/join/group-by state approaches local RAM limits, or when the query must process
data that cannot be safely materialized on a single machine. Query shape matters too:
high-cardinality group-bys, large joins, and large output transformations are stronger
reasons to move to Spark/Dataproc than small aggregations returning only a few rows.

In [10]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
The Pandas PyArrow dtype backend was not universally better than the default backend.
It improved runtime for q1 and q3, but was slower for q2, and it did not reduce measured
memory in this notebook.

For q1, Pandas default took about 0.1421s and PyArrow took about 0.0925s. For q3,
default took about 0.1680s and PyArrow took about 0.1377s. However, for q2, default
was faster at about 0.1645s compared with about 0.1898s for PyArrow. Peak memory was
slightly higher with PyArrow in all three cases: for example q1 increased from about
409.3 MB to 422.5 MB, q2 from 431.9 MB to 454.0 MB, and q3 from 458.5 MB to 461.9 MB.

Overall, PyArrow dtypes helped some Parquet/string-heavy or IO-related operations, but
the benefit depended on query shape.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

The Pandas PyArrow dtype backend was not universally better than the default backend.
It improved runtime for q1 and q3, but was slower for q2, and it did not reduce measured
memory in this notebook.

For q1, Pandas default took about 0.1421s and PyArrow took about 0.0925s. For q3,
default took about 0.1680s and PyArrow took about 0.1377s. However, for q2, default
was faster at about 0.1645s compared with about 0.1898s for PyArrow. Peak memory was
slightly higher with PyArrow in all three cases: for example q1 increased from about
409.3 MB to 422.5 MB, q2 from 431.9 MB to 454.0 MB, and q3 from 458.5 MB to 461.9 MB.

Overall, PyArrow dtypes helped some Parquet/string-heavy or IO-related operations, but
the benefit depended on query shape.